In [2]:
import os
import unicodedata
from google.colab import drive

# 구글 드라이브 마운트
drive.mount('/content/drive')

# 한글 경로 오류 방지 함수
def normalize_path(path):
    return unicodedata.normalize('NFC', path).strip()

Mounted at /content/drive


## 데이터 전처리: COCO JSON → YOLO 형식 변환

- 제공된 COCO 형식 JSON을 YOLO 학습용 형식으로 변환
- train / val = 90 : 10 비율로 분리
- 변환 결과는 구글 드라이브 `yolo_dataset/` 폴더에 저장
- `dataset.yaml` 파일도 자동 생성

In [3]:
# 경로 설정
TRAIN_JSON_PATH = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/dataset/merged_annotations_train_final.json")
TRAIN_IMG_DIR   = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/dataset/train_images")

In [5]:
# 전처리: COCO JSON -> YOLO 형식 변환
import json
import shutil
from pathlib import Path
from collections import defaultdict
from sklearn.model_selection import train_test_split

# YOLO용 폴더 구조 생성
YOLO_ROOT      = normalize_path("/content/drive/MyDrive/data/초급_프로젝트/yolo_dataset/")
YOLO_TRAIN_IMG = os.path.join(YOLO_ROOT, "images", "train")
YOLO_VAL_IMG   = os.path.join(YOLO_ROOT, "images", "val")
YOLO_TRAIN_LBL = os.path.join(YOLO_ROOT, "labels", "train")
YOLO_VAL_LBL   = os.path.join(YOLO_ROOT, "labels", "val")

for d in [YOLO_TRAIN_IMG, YOLO_VAL_IMG, YOLO_TRAIN_LBL, YOLO_VAL_LBL]:
    os.makedirs(d, exist_ok=True)

# COCO JSON 로드
with open(TRAIN_JSON_PATH, "r", encoding="utf-8") as f:
    coco = json.load(f)

# 매핑 준비
id_to_img   = {img["id"]: img for img in coco["images"]}
unique_cats = sorted(set(ann["category_id"] for ann in coco["annotations"]))
cat_to_yolo = {cid: i for i, cid in enumerate(unique_cats)}
img_to_anns = defaultdict(list)
for ann in coco["annotations"]:
    img_to_anns[ann["image_id"]].append(ann)

# train / val 분리 (90:10)
train_ids, val_ids = train_test_split(list(id_to_img.keys()), test_size=0.1, random_state=42)
print(f"train: {len(train_ids)}장 | val: {len(val_ids)}장")

# 변환 함수
def convert_and_save(img_ids, img_dir_dst, lbl_dir_dst):
    for img_id in img_ids:
        img_info  = id_to_img[img_id]
        file_name = img_info["file_name"]
        W, H      = img_info["width"], img_info["height"]

        src = os.path.join(TRAIN_IMG_DIR, file_name)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(img_dir_dst, file_name))

        lines = []
        for ann in img_to_anns.get(img_id, []):
            x, y, w, h = ann["bbox"]
            lines.append(
                f"{cat_to_yolo[ann['category_id']]} "
                f"{(x + w/2)/W:.6f} {(y + h/2)/H:.6f} "
                f"{w/W:.6f} {h/H:.6f}"
            )

        with open(os.path.join(lbl_dir_dst, Path(file_name).stem + ".txt"), "w") as f:
            f.write("\n".join(lines))

convert_and_save(train_ids, YOLO_TRAIN_IMG, YOLO_TRAIN_LBL)
convert_and_save(val_ids,   YOLO_VAL_IMG,   YOLO_VAL_LBL)

# dataset.yaml 생성
id_to_name  = {cat["id"]: cat["name"] for cat in coco["categories"]}
class_names = [id_to_name[cid] for cid in unique_cats]

yaml_path = os.path.join(YOLO_ROOT, "dataset.yaml")
with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(f"path: {YOLO_ROOT}\ntrain: images/train\nval: images/val\n\nnc: {len(unique_cats)}\nnames: {class_names}")

print(f"전처리 완료! dataset.yaml 저장됨: {yaml_path}")

train: 1340장 | val: 149장
전처리 완료! dataset.yaml 저장됨: /content/drive/MyDrive/data/초급_프로젝트/yolo_dataset/dataset.yaml
